# CSC5515 G14 — Kaggle run notebook
Settings → Accelerator: **GPU T4 ×2** (one GPU is used), Internet **on**.
Add-ons → Secrets: `HF_TOKEN` (Llama-3.1 is gated) and `GROQ_API_KEY`.
Add as inputs: a dataset with this code folder and one with `data/processed/` (output of `prepare_data.py`).

In [ ]:
# paths: change to your dataset names
CODE = '/kaggle/input/csc5515-g14-code'
DATA = '/kaggle/input/csc5515-g14-data'
import os, shutil
shutil.copytree(CODE, '/kaggle/working/g14', dirs_exist_ok=True)
os.makedirs('/kaggle/working/g14/data', exist_ok=True)
shutil.copytree(DATA, '/kaggle/working/g14/data/processed', dirs_exist_ok=True)
%cd /kaggle/working/g14
!ls data/processed

In [ ]:
# torch is preinstalled on Kaggle; install the rest
!pip install -q -U transformers peft bitsandbytes accelerate sentence-transformers sacrebleu groq
!pip install -q git+https://github.com/csebuetnlp/normalizer  # for BanglaT5 (optional)
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
sec = UserSecretsClient()
os.environ['HF_TOKEN'] = sec.get_secret('HF_TOKEN')
try:
    os.environ['GROQ_API_KEY'] = sec.get_secret('GROQ_API_KEY')
except Exception:
    print('no GROQ_API_KEY secret')

## Smoke test (≈5 min): 20 rows zero-shot and 10 training steps
Run this first in every new session.

In [ ]:
!python generate.py --model qwen7b --mode zeroshot --limit 20 --system smoke --run p0 --splits val
!head -3 outputs/smoke/p0/val.jsonl
!python train_qlora.py --model qwen7b --max_steps 10 --val_subset 20 --predict_splits --name smoke_train

## Week 7 — baselines and prompting (validation)

In [ ]:
!bash scripts/week7_baselines_prompting.sh

## Week 8 — QLoRA main runs (one model per session; ~3 GPU-h each)

In [ ]:
!bash scripts/week8_qlora_main.sh qwen7b

In [ ]:
# next session
# !bash scripts/week8_qlora_main.sh llama8b

## Save results
Commit the notebook ("Save & Run All" or "Quick save") so that `/kaggle/working` (outputs/, checkpoints/) is kept as notebook output, then attach that output as an input in the next session and copy `outputs/` and `checkpoints/` back into `/kaggle/working/g14`.

In [ ]:
!python evaluate.py --split val
!cat results/val/main_table.md
!zip -qr /kaggle/working/g14_outputs.zip outputs checkpoints/*/log.json results